# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Ollama** para ejecutar Llama directamente en la GPU gratuita de Colab, sin necesidad de API key, cuentas externas ni infraestructura propia.

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab, instalar **Ollama** (el programa que descarga y ejecuta los modelos) y descargar los modelos que vamos a usar. La librería `ollama` es el cliente oficial en Python para hablar con ese servidor.

### **ACTIVAR LA GPU**

En el menú de Colab ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución** y selecciona **T4 GPU**. Luego ejecuta la celda para confirmar que Colab te asignó la GPU.

In [ ]:
# Verificar que Colab asignó una GPU

!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


### **INSTALAR OLLAMA**

A diferencia de una API en la nube, aquí **el modelo corre dentro de tu propia sesión de Colab**: no hay *API key* ni *Secrets*. Ollama se instala en la máquina virtual, se arranca como un servidor local (`localhost:11434`) y desde Python le enviamos los prompts.

**IMPORTANTE:** la sesión de Colab es temporal. Si se reinicia o se desconecta, hay que volver a ejecutar las celdas de instalación, servidor y descarga de modelos.

In [ ]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)

!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado.")

Ollama instalado.


In [ ]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda

import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version

Servidor de Ollama listo.
ollama version is 0.35.1


### **DESCARGAR LOS MODELOS**

Vamos a trabajar con dos tamaños de Llama: **Llama 3.2 de 3B** (ligero) y **Llama 3.1 de 8B** (más grande). Guardamos los nombres en variables para usar *exactamente la misma etiqueta* al descargar y al llamar al modelo; si no coinciden carácter por carácter, Ollama devuelve error.

In [ ]:
# Descargar los dos modelos Llama (la primera vez tarda unos minutos)

MODELO_LIGERO = "llama3.2:3b"
MODELO_GRANDE = "llama3.1:8b"

!ollama pull {MODELO_LIGERO}
!ollama pull {MODELO_GRANDE}
!ollama list



NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    
llama3.2:3b    a80c4f17acd5    2.0 GB    Less than a second ago    


### **CLIENTE DE PYTHON**

In [ ]:
# Instalar el cliente de Ollama y conectarlo al servidor local

!pip install ollama -q

import ollama
import time

client = ollama.Client(host="http://localhost:11434")
print("Cliente de Ollama inicializado correctamente.")

Cliente de Ollama inicializado correctamente.


## **DE PALABRAS A TOKENS**

### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**.

Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [ ]:
# Definir y enviar un prompt a Llama en Ollama y mostrar la respuesta generada

prompt = "¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?"

response = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": prompt}]
)

print(prompt)
print(response.message.content)

¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?
La RAM (Random Access Memory) y el almacenamiento en una computadora son dos componentes fundamentales que funcionan juntos para mejorar el rendimiento y la eficiencia de la computadora. A continuación, te explicaré las diferencias entre ambas:

**RAM (Random Access Memory)**

La RAM es una memoria de acceso aleatorio, que significa que el sistema puede acceder a cualquier ubicación de la memoria de manera rápida y eficiente. La RAM es una memoria volátil, lo que significa que sus datos se pierden cuando se apaga la computadora. La RAM se utiliza para almacenar datos y programas que se están ejecutando actualmente.

**Almacenamiento en una computadora**

El almacenamiento en una computadora se refiere a los dispositivos que contienen datos permanentes que se pueden acceder y recuperar incluso después de que la computadora se apaga. El almacenamiento puede ser de diferentes tipos, como:

*   Disco duro (HD)
*   

In [ ]:
print(response.model_dump_json(indent=2))  # para inspeccionar la respuesta completa

{
  "model": "llama3.2:3b",
  "created_at": "2026-10-05T23:22:55.565092138Z",
  "done": true,
  "done_reason": "stop",
  "total_duration": 13151454069,
  "load_duration": 7247628628,
  "prompt_eval_count": 43,
  "prompt_eval_duration": 128392000,
  "eval_count": 466,
  "eval_duration": 5771927000,
  "message": {
    "role": "assistant",
    "content": "La RAM (Random Access Memory) y el almacenamiento en una computadora son dos componentes fundamentales que funcionan juntos para mejorar el rendimiento y la eficiencia de la computadora. A continuación, te explicaré las diferencias entre ambas:\n\n**RAM (Random Access Memory)**\n\nLa RAM es una memoria de acceso aleatorio, que significa que el sistema puede acceder a cualquier ubicación de la memoria de manera rápida y eficiente. La RAM es una memoria volátil, lo que significa que sus datos se pierden cuando se apaga la computadora. La RAM se utiliza para almacenar datos y programas que se están ejecutando actualmente.\n\n**Almacenamient

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de Ollama trae el conteo de tokens de entrada (`prompt_eval_count`) y de salida (`eval_count`). Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [ ]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta

tokens_prompt = response.prompt_eval_count
tokens_respuesta = response.eval_count

print("Tokens del prompt:", tokens_prompt)
print("Tokens de la respuesta:", tokens_respuesta)
print("Tokens totales:", tokens_prompt + tokens_respuesta)
# Con Ollama no se factura: los tokens miden cuánto trabajo hizo el modelo y cuánto de su contexto usaste

Tokens del prompt: 43
Tokens de la respuesta: 466
Tokens totales: 509


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [ ]:
# Medir el tiempo de respuesta de Llama para el mismo prompt

inicio = time.time()
response_tiempo = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio
# duracion_ms = duracion * 1000  # si prefieres reportarlo en milisegundos

print(f"Tiempo de respuesta: {duracion:.2f} segundos")

Tiempo de respuesta: 8.93 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama que cabe en la GPU gratuita de Colab (8B) y compara tiempo de respuesta y calidad contra el modelo ligero (3B).

Ojo: la primera vez que se usa un modelo, Ollama tiene que **cargarlo en la memoria de la GPU**. Ese tiempo de carga viene en `load_duration` y conviene separarlo para que la comparación sea justa.

In [ ]:
# Repetir la llamada con un modelo Llama más grande y comparar tiempo y calidad

inicio = time.time()
response_grande = client.chat(
    model=MODELO_GRANDE,
    messages=[{"role": "user", "content": prompt}]
)
duracion_grande = time.time() - inicio
carga_grande = response_grande.load_duration / 1e9  # nanosegundos -> segundos

tokens_ligero = response.prompt_eval_count + response.eval_count
tokens_grande = response_grande.prompt_eval_count + response_grande.eval_count

print(f"Modelo ligero ({MODELO_LIGERO}): {duracion:.2f} s — {tokens_ligero} tokens")
print(f"Modelo grande ({MODELO_GRANDE}): {duracion_grande:.2f} s — {tokens_grande} tokens (de los cuales {carga_grande:.2f} s fueron carga del modelo)")
print("\nRespuesta del modelo grande:\n", response_grande.message.content)

Modelo ligero (llama3.2:3b): 8.93 s — 509 tokens
Modelo grande (llama3.1:8b): 16.78 s — 185 tokens (de los cuales 11.89 s fueron carga del modelo)

Respuesta del modelo grande:
 La diferencia entre la RAM y el almacenamiento en una computadora es la siguiente:

- La RAM (Memoria de Acceso Aleatorio) es la memoria de trabajo de la computadora. Algunos programas y archivos se cargan en la memoria RAM cuando la computadora está encendida. La RAM tiene una capacidad menor a la del almacenamiento y es volátil, por lo que cuando la computadora se apaga se borran los datos.

- El almacenamiento es el lugar donde se guardan los datos de manera permanente, como archivos, programas, imágenes, etc. Algunos ejemplos de almacenamiento son los discos duros, discos rígidos, unidades USB y SSD (Unidad de Estado Sólido).


## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

Una vez visto el ***Hands-On: Fundamentos de LLMs***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño **comparador que envíe 3 preguntas** reales a Llama y registre, para cada una, el **tiempo de respuesta** y el **número de tokens** consumidos.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key:** Prepara el entorno de configuración del entorno con **Ollama** activando la **T4 GPU** e inicializa el cliente.

In [1]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)
!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado.")

Ollama instalado.


In [2]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda
import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

Servidor de Ollama listo.


In [3]:
# Descargar el modelo ligero de Llama e inicializar el cliente
MODELO_LIGERO = "llama3.2:3b"

!ollama pull {MODELO_LIGERO}

!pip install ollama -q
import ollama
client = ollama.Client(host="http://localhost:11434")
print("Cliente de Ollama inicializado correctamente.")


Cliente de Ollama inicializado correctamente.


**2. Consulta la primera pregunta:**

* Construye una lista llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).
*  Envía la primera pregunta a Llama usando el modelo más ligero disponible en Groq. Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

In [4]:
# Definir la lista de preguntas
preguntas = [
    "¿Cuál es la diferencia entre grados Celsius y Kelvin?",
    "¿Por qué el cero absoluto es -273.15 °C?",
    "¿Cómo convierto grados Fahrenheit a Celsius?"
]

In [5]:
# Consultar la pregunta 1 y guardar el resultado en resultado_1
def consultar(pregunta):
    inicio = time.time()
    respuesta = client.chat(
        model=MODELO_LIGERO,
        messages=[{"role": "user", "content": pregunta}]
    )
    duracion = time.time() - inicio
    return {
        "pregunta": pregunta,
        "respuesta": respuesta.message.content,
        "tiempo_s": round(duracion, 2),
        "tokens": respuesta.prompt_eval_count + respuesta.eval_count
    }

resultado_1 = consultar(preguntas[0])
print(resultado_1["tiempo_s"], "s |", resultado_1["tokens"], "tokens")

101.03 s | 553 tokens


**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

In [6]:
# Consultar la pregunta 2 y guardar el resultado en resultado_2
resultado_2 = consultar(preguntas[1])
print(resultado_2["tiempo_s"], "s |", resultado_2["tokens"], "tokens")

6.6 s | 549 tokens


In [7]:
# Consultar la pregunta 3 y guardar el resultado en resultado_3
resultado_3 = consultar(preguntas[2])
print(resultado_3["tiempo_s"], "s |", resultado_3["tokens"], "tokens")

1.71 s | 161 tokens


**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

In [8]:
# Definir la lista resultados y agregar los tres diccionarios
resultados = []
resultados.append(resultado_1)
resultados.append(resultado_2)
resultados.append(resultado_3)

**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [9]:
# Mostrar la tabla final de resultados
for r in resultados:
    print("PREGUNTA:", r["pregunta"])
    print("TIEMPO:", r["tiempo_s"], "s | TOKENS:", r["tokens"])
    print("RESPUESTA:", r["respuesta"])
    print("-" * 60)

PREGUNTA: ¿Cuál es la diferencia entre grados Celsius y Kelvin?
TIEMPO: 101.03 s | TOKENS: 553
RESPUESTA: La diferencia entre grados Celsius y Kelvin se refiere a la unidad de medida del temperamento. A continuación te presento los detalles sobre ambas escalas:

Grados Celsius:
La escala de grados Celsius es una escala de temperaturas que se utiliza comúnmente en la práctica cotidiana y científica. En esta escala, el punto de congelación del agua es de 0 grados Celsius (0°C) y el punto de ebullición del agua es de 100 grados Celsius (100°C). Esta escala es lineal y se utiliza para medir temperaturas en una amplia gama de aplicaciones.

Kelvin:
La escala de Kelvin, por otro lado, es una escala de temperaturas absolutas que se utiliza en la física y la química. En esta escala, el punto de congelación del agua es de 273,15 Kelvin (K) y el punto de ebullición del agua es de 373,15 K. La escala de Kelvin es aditiva, lo que significa que se puede sumar y restar temperaturas sin perder la pro